# 03 실전 반도체 공정 데이터 분석 · 5~6. 특성 선택 · 핵심 EDA

- 강의 페이지: `Web/강좌/03_실전_반도체_공정_데이터분석/실전_반도체_공정_데이터분석_강의자료.html` → 목차 **특성 선택 · 핵심 EDA**
- SelectKBest로 상위 20개 센서를 고르고 박스플롯·히트맵으로 살핍니다.
- `fab.csv`가 이 노트북과 같은 폴더에 있어야 합니다.
- 위에서 아래로 순서대로 실행하세요. 다른 노트북의 변수를 사용하지 않으므로 이 파일만 열어도 실행됩니다.

### 5단계 · 특성 선택 NEW
- 원본 코드: `Python — 상위 K개 센서 자동 선택`
- 설명: 통계 검정으로 예측에 중요한 상위 센서 컬럼을 선택합니다.
- 수업 메모: 중요한 K개 센서만 추리기 (ANOVA F-검정)

### 준비 · 1~4단계 코드 실행

앞 단계 코드를 그대로 모아 한 번에 실행합니다. 출력은 앞 노트북과 같습니다.

In [ ]:
# ── 1단계 ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif   # ⭐ NEW
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# 그래프 한글 설정 (Mac은 'AppleGothic', Colab·리눅스는 'NanumGothic')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ── 2단계 ──
df = pd.read_csv('fab.csv')

# ── 3단계 결측값 처리 ──
# 1️⃣ 컬럼별 결측률 계산
miss_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(miss_pct.sort_values(ascending=False).head(10))

# 2️⃣ 결측률 50% 초과 컬럼 제거
THRESHOLD = 50.0
cols_to_drop = miss_pct[miss_pct > THRESHOLD].index.tolist()
print(f"🗑️ 제거할 컬럼: {len(cols_to_drop)}개")
df = df.drop(columns=cols_to_drop)

# 3️⃣ 남은 결측치는 중앙값으로 채우기 (타겟 제외!)
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Pass_Fail')   # ⚠️ 타겟은 절대 채우지 말 것!

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
print(f"✅ 결측 처리 완료. 남은 결측: {df.isnull().sum().sum()}")

# ── 4단계 분산 0 제거 ──
# 분산 = 0 컬럼 + 거의 0인 컬럼 모두 제거
variances = df[numeric_cols].var()

constant_cols    = variances[variances == 0].index.tolist()
near_constant    = variances[(variances > 0) & (variances < 1e-6)].index.tolist()

print(f"분산 0  컬럼: {len(constant_cols)}개")
print(f"분산 ≈0 컬럼: {len(near_constant)}개")

to_drop_var = constant_cols + near_constant
df = df.drop(columns=to_drop_var)
numeric_cols = [c for c in numeric_cols if c not in to_drop_var]

print(f"✅ 사용 가능한 센서: {len(numeric_cols)}개")


In [ ]:
# 타겟을 0/1로 변환: 불량(1)을 양성 클래스로
y = (df['Pass_Fail'] == 1).astype(int)
X_all = df[numeric_cols].copy()

# 상위 K=20 개 센서 자동 선택
K = 20
selector = SelectKBest(score_func=f_classif, k=K)
selector.fit(X_all, y)

f_scores   = pd.Series(selector.scores_, index=numeric_cols)\
               .replace([np.inf, -np.inf], np.nan).dropna()
top_k_cols = f_scores.sort_values(ascending=False).head(K).index.tolist()

print(f"🏆 선택된 상위 {K}개 센서:")
for i, col in enumerate(top_k_cols, 1):
    print(f"  {i:2d}. {col}  F={f_scores[col]:6.1f}")

### 5단계 · 특성 선택 NEW
- 원본 코드: `Python — 상위 K 시각화`
- 설명: 통계 검정으로 예측에 중요한 상위 센서 컬럼을 선택합니다.
- 수업 메모: 중요한 K개 센서만 추리기 (ANOVA F-검정)

In [ ]:
top = f_scores.sort_values(ascending=False).head(K)
plt.figure(figsize=(10, 7))
plt.barh(top.index[::-1], top.values[::-1], color='#3498db')
plt.xlabel('F-값 (클수록 유용)')
plt.title('상위 K개 센서 F-점수')
plt.tight_layout()
plt.show()

## 6단계 · 핵심 변수 EDA

### 6단계 · 핵심 변수 EDA
- 원본 코드: `Python — 상위 6개 박스플롯`
- 설명: 통계 검정으로 예측에 중요한 상위 센서 컬럼을 선택합니다.
- 수업 메모: 상위 K개로 줄였으니 이제 시각화 가능!

In [ ]:
top6 = top_k_cols[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(top6):
    sns.boxplot(x='Pass_Fail', y=col, data=df,
                hue='Pass_Fail',
                palette={-1: '#2ecc71', 1: '#e74c3c'},
                legend=False, ax=axes[i], width=0.5)
    axes[i].set_title(f'{col}  (F={f_scores[col]:.1f})')
    axes[i].set_xticks([0, 1], ['정상(-1)', '불량(+1)'])

plt.suptitle('상위 6개 센서 — 정상 vs 불량 분포')
plt.tight_layout()
plt.show()

### 6단계 · 핵심 변수 EDA
- 원본 코드: `Python — 상위 K 상관관계 히트맵`
- 설명: 통계 검정으로 예측에 중요한 상위 센서 컬럼을 선택합니다.
- 수업 메모: 상위 K개로 줄였으니 이제 시각화 가능!

In [ ]:
corr_top = df[top_k_cols + ['Pass_Fail']].corr().round(2)

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_top, dtype=bool))
sns.heatmap(corr_top, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, mask=mask,
            linewidths=0.5)
plt.title('상위 센서 간 상관관계 + Pass_Fail')
plt.show()

## 마무리

- F-값이 큰 센서는 정상/불량 평균 차이가 뚜렷한 센서입니다. 원인이 아니라 **추가 점검 단서**로 읽으세요.
- 여기서는 결측 처리·특성 선택을 전체 데이터로 했습니다. 실무에서는 학습 데이터에만 맞춰야 합니다(데이터 누수, 보강 실습 참고).